# Microsoft Fabric — End-to-end SDLC + Purview Governance Demo

**Audience:** Customer architecture review covering Data Engineering, Reporting, and Analytics teams.

**Goal:** Walk through a working CI/CD pipeline (ADO + GitHub) that promotes Fabric items dev → uat → prod across three teams, and shows Microsoft Purview discovering everything via a scheduled scan.

**Stack used in this demo**

| Layer | Tooling |
|---|---|
| Source control | Azure DevOps Repos + GitHub (mixed, mirroring real-world team preferences) |
| Pipelines | ADO Pipelines using the **Microsoft Fabric DevOps Pipelines** marketplace extension (`ms-fabric.fabric-devops-pipelines`) + GitHub Actions mirror |
| Deployment lib | [`fabric-cicd`](https://microsoft.github.io/fabric-cicd) Python package |
| Identity | Service Principal in Entra, secret in Azure Key Vault, exposed via ADO variable group |
| Governance | Microsoft Purview Unified Catalog (governance domains + Fabric scan) |

> ADO org used for the live demo: <https://dev.azure.com/ncheruvu0468/NagaDevops>

## 1. The 3-team topology (open the diagram)

Open `diagrams/fabric-sdlc-purview.drawio` in <https://app.diagrams.net> for the full visual. Summary in ASCII below for inline reference.

```
 ADO repo (DE) ──▶ pipeline-de ──▶ de-dev ⇄ Git ──▶ de-uat ──▶┐
 GH  repo (RP) ──▶ pipeline-reporting ─▶ reporting-dev ⇄ Git ─▶ reporting-uat ─▶ analytics-prod (shared) ──▶ Purview
 GH  repo (AN) ──▶ pipeline-analytics ─▶ analytics-dev  ⇄ Git ─▶ analytics-uat ─▶┘
```

### Customer questions — answered

| Question | Answer |
|---|---|
| 1 master pipeline / 2 / 3? | **3, one per team**, all merging into the shared Prod workspace. |
| How many Fabric capacities? | **2 minimum** (non-prod F8, prod F32+); split DE prod from BI prod only when noisy-neighbour shows up. |
| Capacity alignment | By **environment first**, by **workload type at prod tier**. Not by team. |
| Table types | Medallion: bronze (raw) → silver (typed, contracted) → gold (star schema). |
| Data contracts | YAML per gold table, validated in CI by `scripts/validate_contracts.py`. |
| Purview hookup | Register Fabric tenant as data source, weekly recurring scan, governance domains for each business team. |

## 2. Bootstrap the local environment

In [ ]:
# Run once. Safe to skip if requirements.txt is already installed.
%pip install -q -r ../requirements.txt

In [ ]:
import os, sys, subprocess, json, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path('..').resolve()
load_dotenv(ROOT / '.env')
print('Tenant:', os.getenv('TENANT_ID', '<not set>'))
print('Purview account:', os.getenv('PURVIEW_ACCOUNT', '<not set>'))
print('Fabric prod ws:', os.getenv('FABRIC_PROD_WS', '<not set>'))

## 3. Dry-run the Purview governance bootstrap

This shows every REST call the script will make — no live calls yet. Confirm the URLs and bodies look right before flipping off `--dry-run`.

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'purview_fabric_governance.py'), '--dry-run'],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr, file=sys.stderr)

## 4. Live Purview run (requires populated `.env` and SPN permissions)

Uncomment and run when ready. Idempotent — re-running is safe.

In [ ]:
# subprocess.run([sys.executable, str(ROOT / 'scripts' / 'purview_fabric_governance.py')], check=True, cwd=ROOT)

## 5. Verify the ADO setup against your org

Calls the Azure DevOps REST API to list pipelines + variable groups in `NagaDevops`. Requires a PAT in `ADO_PAT` env var with **Read** on Build + Library.

In [ ]:
import requests, base64

ADO_ORG = os.getenv('ADO_ORG', 'ncheruvu0468')
ADO_PROJECT = os.getenv('ADO_PROJECT', 'NagaDevops')
PAT = os.getenv('ADO_PAT')

if not PAT:
    print('Set ADO_PAT to a PAT with Build (Read) + Library (Read).')
else:
    auth = base64.b64encode(f':{PAT}'.encode()).decode()
    headers = {'Authorization': f'Basic {auth}'}
    base = f'https://dev.azure.com/{ADO_ORG}/{ADO_PROJECT}/_apis'

    pipes = requests.get(f'{base}/pipelines?api-version=7.1', headers=headers, timeout=30).json()
    print('Pipelines:')
    for p in pipes.get('value', []):
        print(f"  • [{p['id']}] {p['name']}  ({p['folder']})")

    vgs = requests.get(
        f'https://dev.azure.com/{ADO_ORG}/_apis/distributedtask/variablegroups?project={ADO_PROJECT}&api-version=7.1-preview.2',
        headers=headers, timeout=30,
    ).json()
    print('\nVariable groups:')
    for v in vgs.get('value', []):
        print(f"  • [{v['id']}] {v['name']}  type={v.get('type','Vsts')}")

## 6. Push the pipeline templates into your ADO repo

From your repo root after cloning, copy the files:

```powershell
Copy-Item ado-pipelines\*.yml .\
Copy-Item -Recurse ado-pipelines\templates .\
git checkout -b feat/fabric-cicd
git add . ; git commit -m 'feat: fabric CI/CD pipelines for DE/Reporting/Analytics'
git push -u origin feat/fabric-cicd
```

Then in ADO portal:
1. **Pipelines → New pipeline** → Azure Repos Git → existing YAML → `pipeline-de.yml`. Repeat for the other two.
2. **Pipelines → Library** → create the two variable groups exactly as named in `templates/install-fabric-tools.yml`.
3. **Pipelines → Environments** → create `dev`, `test`, `prod`; add a manual approval check on `test` and `prod`.

## 7. Validate a sample data contract against UAT

Same script the CI step runs. Edit `contracts/gold/fact_sales.yml` to introduce a type drift and watch this fail.

In [ ]:
# subprocess.run(
#     [sys.executable, str(ROOT/'scripts'/'validate_contracts.py'), str(ROOT/'contracts/gold/*.yml')],
#     env={**os.environ, 'FABRIC_SQL_ENDPOINT': os.getenv('FABRIC_SQL_ENDPOINT_UAT','')},
#     check=True, cwd=ROOT,
# )

## 8. What to walk the customer through (suggested 30-min agenda)

| Min | Section | Asset |
|---:|---|---|
| 0–5 | Open the topology diagram, confirm the 3-team / shared-prod model. | `diagrams/fabric-sdlc-purview.drawio` |
| 5–10 | Walk the trade-off matrix on pipeline count + capacity sizing. | `docs/sdlc-reference-architecture.md` §3, §4 |
| 10–15 | Show a YAML pipeline live; explain `fabric-cicd` parameter swap. | `ado-pipelines/pipeline-de.yml`, `parameter.yml` |
| 15–20 | Show a data contract + the validator failing on drift. | `contracts/gold/fact_sales.yml` |
| 20–28 | Run the Purview script (dry-run, then live). Show domains in the Purview portal. | `scripts/purview_fabric_governance.py` |
| 28–30 | Recap the RACI and next steps. | §8 of architecture doc |